In [12]:
import numpy as np
import jax.numpy as jnp
import matplotlib as mpl
import matplotlib.pyplot as plt
import pickle
import immunowave as iw
from immunowave_paper_utils import style_axes, colors, fontsize, linewidth, rc_params
from diffrax import SaveAt
from scipy.optimize import curve_fit
from functools import partial

In [2]:
linewidth = 3
fontsize = 24
markersize = 12
markeredgewidth = 2
rc_params["axes.linewidth"] = linewidth
rc_params["font.size"] = fontsize
mpl.rcParams.update(rc_params)
mpl.rcParams["pdf.fonttype"] = 42
sim_color = np.array([94, 45, 144]) / 255


In [3]:
path_to_data = r'/home/brandon/Documents/Code/immunowave/data/2025_07_28_2D'

### Define model

In [4]:
class State(iw.State):
    u: iw.ScalarField

Our dynamical variable $u$ satisfies
$$
\partial_t u = D\nabla^2 u + f(u) + 2 I \delta(\vec{x})
$$

where
$$
f(u) = k\frac{u^n}{K^n_D + u^n} - \gamma u
$$
with $I, K_D > 0$, and $n > 1$.


In [5]:
class Model(iw.Model):
    KD: float
    n: float
    I: float
    k: float = 1.0
    γ: float = 1.0
    D: float = 1.0
    # Bandwidth adjustment for Dirac delta approximation (grid units)
    bw_adjust: float = 0.5

    def f(self, u):
        KD, n, k, γ = self.KD, self.n, self.k, self.γ
        return k * u.hill(KD, n) - γ * u

    def delta(self, u):
        """Helper function to approximate a Dirac delta as a Gaussian on the same grid as u."""
        σ = self.bw_adjust * u.h

        def gaussian(*coords):
            squared_dist = sum(coord**2 for coord in coords)
            return jnp.exp(-squared_dist / (2 * σ**2))

        result = iw.ScalarField(u.values.shape, u.lb, u.h, fn=gaussian)
        result /= result.integral()
        return result

    def __call__(self, t, state: State, args=None):
        u = state.u
        Δu = u.laplacian(bc="neumann")
        f = self.f
        I = self.I
        δ = self.delta(u)
        D = self.D

        dudt = D * Δu + f(u) + 2 * I * δ

        return State(dudt)

### Analytical $I_c$

In dimensionless units, for $K_D\ll1$, the unstable fixed point is approximately $u_1 \approx K^{n/(n-1)}_D$, and the critical stimulus is
$$
I_c \approx \sqrt{\frac{n-1}{n+1}}\,u_1
$$


In [6]:
def u1(KD, n):
    return KD ** (n / (n - 1))


def Ic(KD, n, k=1.0, D=1.0, γ=1.0):
    return ((n - 1) / (n + 1)) ** 0.5 * u1(KD * γ / k, n) * (D / γ) ** 0.5 * k

## Plot everything including schematics

In [7]:
plt.close("all")

In [84]:
%matplotlib qt
f, axs = plt.subplots(2, 3, figsize=(18.97, 10.79))

### comparison with numerics --- KD

In [85]:
def KD_scaling_theory(KD, c, n):
    return c * KD ** (n / (n-1))

def fit_Ic_vs_KD_prefactor(KDs, Ics, n):
    p0 = [1,]
    popt, pcov = curve_fit(partial(KD_scaling_theory, n=n), KDs, Ics, p0=p0)
    return popt[0]

In [86]:
"""plot"""
# load the data
with open(
    path_to_data + "/KD_sweep.pkl",
    "rb",
) as f:
    KD_sweep_dict = pickle.load(f)
KDs = KD_sweep_dict["KD_grid"]
Ic_wave = KD_sweep_dict["Ic_grid"]
hill_coefficient = 4

ax = axs[1, 0]
ax.clear()
prefactor = fit_Ic_vs_KD_prefactor(KDs, Ic_wave, hill_coefficient)
KDs_theory = np.logspace(np.log10(np.min(KDs)), np.log10(np.max(KDs)), 1000)
tissue_theory = scaling_theory(KDs_theory, prefactor, hill_coefficient)
ax.plot(
    KDs_theory,
    tissue_theory,
    "-",
    linewidth=linewidth,
    color=colors["wave"],
    label="$\sim K_D^{n/(n-1)}$",
)


ax.plot(
    KDs,
    Ic_wave,
    marker="o",
    markersize=markersize,
    markerfacecolor=np.concatenate((sim_color, [0.0])),
    markeredgewidth=markeredgewidth,
    linestyle="none",
    markeredgecolor=sim_color,
    label="numerics",
)

ax.legend(fontsize=fontsize, loc='upper left')
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylabel(r"$I^{wave}_{c}$ (a.u.)", fontsize=fontsize)
ax.set_xlabel(
    "concentration scale \nof positive feedback, $K_D$ (a.u.)", fontsize=fontsize
)
ax.set_ylim([2e-3, 5e-1])
ax.minorticks_off()
style_axes(ax, fontsize=fontsize)

<>:23: SyntaxWarning: invalid escape sequence '\s'
<>:23: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_586886/2981009009.py:23: SyntaxWarning: invalid escape sequence '\s'
  label="$\sim K_D^{n/(n-1)}$",


<Axes: xlabel='concentration scale \nof positive feedback, $K_D$ (a.u.)', ylabel='$I^{wave}_{c}$ (a.u.)'>

### comparison with numerics--- vary hill coeff

In [87]:
"""plot"""

ns = np.linspace(1.3, 5, 100)
KD = 0.01

# load the data
with open(
    path_to_data + "/n_sweep.pkl",
    "rb",
) as f:
    hill_coeff_sweep_dict = pickle.load(f)
hill_coefficients =  jnp.linspace(1.3, 5.0, 20)#hill_coeff_sweep_dict["n_grid"]
Ic_wave = hill_coeff_sweep_dict["Ic_grid"]

ax = axs[1, 1]
ax.clear()

tissue_theory = Ic(KD, ns)
ax.plot(
    ns,
    tissue_theory / KD ** (ns / (ns - 1)),
    "-",
    linewidth=linewidth,
    color=colors["wave"],
    label="wave theory",
)

ax.plot(
    hill_coefficients,
    Ic_wave / KD ** (hill_coefficients / (hill_coefficients - 1)),
    marker="o",
    markersize=markersize,
    markerfacecolor=np.concatenate((sim_color, [0.0])),
    markeredgewidth=markeredgewidth,
    linestyle="none",
    markeredgecolor=sim_color,
    label="wave numerics",
)


ax.set_xscale("linear")
ax.set_yscale("log")
ax.set_ylabel(r"$I^{wave}_{c}$ (a.u.)", fontsize=fontsize)
ax.set_xlabel("hill coefficient, $n$", fontsize=fontsize)
ax.set_ylim([1e-1, 1e1])

#ax.set_ylim([1e-10, 1e-2])
#ax.set_yticks([1e-10, 1e-6, 1e-2])
ax.minorticks_off()
style_axes(ax, fontsize=fontsize)

<Axes: xlabel='hill coefficient, $n$', ylabel='$I^{wave}_{c}$ (a.u.)'>

### comparision with numerics---vary gamma

In [74]:
def gamma_scaling_theory(gamma, c, n):
    return c * gamma ** (n / (n - 1) - 1)


def fit_Ic_vs_gamma_prefactor(gammas, Ics, n):
    p0 = [1e-4,]
    popt, pcov = curve_fit(partial(gamma_scaling_theory, n=n), gammas, Ics, p0=p0)
    #popt, pcov = curve_fit(gamma_scaling_theory, gammas, Ics, p0=p0)

    return popt[0]


def power_law(x, prefactor, exponent):
    return prefactor * x ** exponent

    
def fit_power_law(x, y):
    p0 = [1e-4, 1]
    popt, pcov = curve_fit(power_law, x, y, p0=p0)

    return popt

In [83]:
"""plot"""

KD = 0.01

# load the data
with open(
    path_to_data + "/gamma_sweep.pkl",
    "rb",
) as f:
    gamma_sweep_dict = pickle.load(f)
gammas = gamma_grid = jnp.linspace(0.1, 1.8, 20)#gamma_sweep_dict["gammas"]
Ic_wave = gamma_sweep_dict["Ic_grid"]

gamma_theory = np.linspace(np.min(gammas), np.max(gammas), 1000)
hill_coefficient = 4

ax = axs[1, 2]
ax.clear()

prefactor = fit_Ic_vs_gamma_prefactor(gammas, Ic_wave, hill_coefficient)
tissue_theory = gamma_scaling_theory(gamma_theory, prefactor, hill_coefficient)
# prefactor, exponent = fit_power_law(gammas, Ic_wave)
# tissue_theory = power_law(gamma_theory, prefactor, exponent)
ax.plot(
    gamma_theory,
    tissue_theory / gamma_theory ** (hill_coefficient / (hill_coefficient - 1) - 1),
    "-",
    linewidth=linewidth,
    color=colors["wave"],
    label="wave theory",
)

ax.plot(
    gammas,
    Ic_wave / gammas ** (hill_coefficient / (hill_coefficient - 1) - 1),
    marker="o",
    markersize=markersize,
    markerfacecolor=np.concatenate((sim_color, [0.0])),
    markeredgewidth=markeredgewidth,
    linestyle="none",
    markeredgecolor=sim_color,
    label="wave numerics",
)

ax.set_xscale("linear")
ax.set_yscale("linear")
ax.set_ylabel(r"$I^{wave}_{c}$ (a.u.)", fontsize=fontsize)
ax.set_xlabel(r"decay rate, $\gamma$ (1/min)", fontsize=fontsize)
ax.set_ylim([3e-3, 6e-3])

#ax.set_ylim([1e-4, 1e-2])
ax.minorticks_off()
style_axes(ax, fontsize=fontsize)

<Axes: xlabel='decay rate, $\\gamma$ (1/min)', ylabel='$I^{wave}_{c}$ (a.u.)'>

In [54]:
Ic_wave

Array([0.00166463, 0.0021222 , 0.00247452, 0.00277648, 0.00304814,
       0.00329941, 0.00353552, 0.00376052, 0.0039767 , 0.0041857 ,
       0.00438859, 0.00458702, 0.004781  , 0.00497154, 0.00515923,
       0.00534439, 0.00552676, 0.00570763, 0.00588644, 0.00606339],      dtype=float64)

In [76]:
exponent

np.float64(0.4825483982432252)

In [169]:
plt.gcf().tight_layout()

In [178]:
#plt.savefig(
#    r"/home/brandon/Documents/Code/immunowave/plots/2025-07-29_numerics_comparision_fig-wave_v4.pdf"
#)